# Actividad individual 2 — Descubriendo patrones de tráfico con datos reales
## TC2035 — Diseño de Redes Neuronales y Aprendizaje Profundo

**Nombre:** Emilio Alonso Sánchez  
**Matrícula:** A00841853  
**Grupo:** *Completar con el grupo correspondiente*  
**Modalidad:** Individual  
**Herramienta:** Google Colab (Python)  
**Reto:** Predicción inteligente de tiempos de verde — SEMEX

### Introducción
Antes de desarrollar una red neuronal capaz de recomendar tiempos de verde (*splits*), necesito entender la información que recibe. En esta actividad analizo los registros del estudio de volúmenes vehiculares de la intersección 01 en Distrito Tec, Monterrey, exploro los intervalos de medición, comparo tres movimientos vehiculares, normalizo sus volúmenes y explico cómo podrían utilizarse como entradas de una futura MLP.

**Alcance:** exploración de datos, sin entrenamiento de redes neuronales. No se calculan tiempos de verde óptimos porque el archivo de conteos no incorpora por sí solo etiquetas verificadas de *splits* recomendados.

**Fuentes:** \`Int. 01 mañana.xlsx\` (datos para los cálculos) y \`Croquis-01.pdf\` (referencia espacial y comparativa de tres periodos). Se conserva la distinción entre **datos del Excel** y **totales publicados en el croquis**.

### Contexto de la intersección según el croquis

El croquis del estudio reporta los siguientes volúmenes durante la **hora de máxima demanda de cada periodo**. No deben confundirse con todos los vehículos observados durante las tres horas de cada levantamiento.

| Periodo | Levantamiento | Hora de máxima demanda | Volumen total |
|---|---|---|---:|
| Mañana | 07:00–10:00 | 08:00–09:00 | 1,771 vehículos |
| Tarde | 12:00–15:00 | 13:45–14:45 | 2,287 vehículos |
| Noche | 17:00–20:00 | 17:15–18:15 | 2,372 vehículos |

En la lámina de la **mañana** se muestran cuatro valores de acceso que suman 1,771: 454, 305, 548 y 464 vehículos. También se representan movimientos alrededor de la rotonda en la zona de Av. Eugenio Garza Sada, Av. Fernando García Roel, Del Estado y el Tecnológico de Monterrey.

**Precaución:** el croquis permite contextualizar los accesos, pero la correspondencia de una columna particular del Excel con una trayectoria del croquis debe verificarse con los encabezados y la codificación original. No se asignan direcciones o giros inventados.

---

## Parte 1. Exploración y comprensión de los datos

### 1.1 Preparación y carga del archivo

Este notebook **intenta descargar** el Excel desde la rama \`main\` de \`A00841853/Redes-Neuronales\`. Si todavía no está disponible en GitHub, permite seleccionarlo con el botón de carga de Colab. En VS Code/Jupyter local también puede leerlo si está en la misma carpeta del notebook.

Si los registros se organizaron con encabezados combinados o varias tablas en una hoja, las siguientes celdas mostrarán las filas sin procesar antes de elegir los movimientos. **No se rellenan con datos simulados**.

In [ ]:
# Bibliotecas: Colab incluye habitualmente estas dependencias
import re
from pathlib import Path
from datetime import datetime, time
from io import BytesIO
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 180)
np.set_printoptions(suppress=True, precision=4)

ARCHIVO = "Int. 01 mañana.xlsx"
URL = "https://raw.githubusercontent.com/A00841853/Redes-Neuronales/main/" + quote(ARCHIVO)

archivo_excel = None
if Path(ARCHIVO).is_file():
    archivo_excel = Path(ARCHIVO)
    print("Excel encontrado localmente:", archivo_excel)
else:
    import requests
    try:
        respuesta = requests.get(URL, timeout=15)
        if respuesta.ok and respuesta.content[:2] == b"PK":
            archivo_excel = BytesIO(respuesta.content)
            print("Excel descargado de GitHub:", URL)
        else:
            print("El archivo Excel aún no está accesible en GitHub (código:", respuesta.status_code, ").")
    except requests.RequestException as exc:
        print("No se pudo acceder a GitHub:", exc)

if archivo_excel is None:
    try:
        from google.colab import files
        print("Selecciona ahora el archivo:", ARCHIVO)
        subidos = files.upload()
        rutas = [nombre for nombre in subidos if nombre.lower().endswith((".xlsx", ".xls"))]
        if not rutas:
            raise FileNotFoundError("No seleccionaste un archivo Excel.")
        archivo_excel = BytesIO(subidos[rutas[0]])
        print("Archivo cargado:", rutas[0])
    except ImportError as exc:
        raise FileNotFoundError(
            "No se encontró el Excel. Pon 'Int. 01 mañana.xlsx' en la carpeta del notebook, "
            "o ejecuta este notebook en Google Colab para subir el archivo."
        ) from exc

libro = pd.ExcelFile(archivo_excel)
print("\nHojas disponibles:", libro.sheet_names)
hojas_crudas = {
    nombre: pd.read_excel(libro, sheet_name=nombre, header=None, dtype=object)
    for nombre in libro.sheet_names
}
for nombre, tabla in hojas_crudas.items():
    print(f"Hoja: {nombre!r} | filas={tabla.shape[0]}, columnas={tabla.shape[1]}")

In [ ]:
# Explorar TODAS las hojas sin asumir en qué fila empieza la tabla
for nombre, tabla in hojas_crudas.items():
    display(Markdown(f"#### Vista inicial: {nombre}"))
    display(tabla.iloc[:20, :min(24, tabla.shape[1])].fillna("").copy())

print("Consejo: identifica las filas con horas y las columnas asociadas a los movimientos.")
print("Conservamos la tabla original para no descartar encabezados especiales.")

### Interpretación 1. Comprendiendo los datos

- **¿Qué representa cada fila?** Las filas que se conservarán en la tabla analítica representan un **intervalo de observación** del archivo, no necesariamente un vehículo individual. Antes de filtrar, el Excel también puede contener filas de títulos, encabezados, sumas y notas.
- **¿Qué variables identifico?** La hora o intervalo de medición, el identificador de cada movimiento y su volumen vehicular. La selección exacta de las columnas se muestra y verifica más adelante; los códigos de movimiento deben contrastarse con el croquis.
- **¿Qué unidades tienen los volúmenes?** Vehículos contados durante el intervalo registrado. Si el intervalo es de 15 minutos, la unidad es *vehículos por intervalo de 15 minutos*; no equivale directamente a vehículos por hora.
- **¿Qué información resulta útil?** La hora, el volumen por movimiento, la composición de las corrientes, el máximo por intervalo y las diferencias entre trayectorias; más adelante se pueden añadir ocupación, velocidad y colas.
- **¿Hay datos faltantes o encabezados especiales?** Se examina la tabla sin procesar antes de hacer conversiones, se contabilizan las celdas vacías y se mantiene \`NaN\` cuando un conteo está ausente. No se sustituyen automáticamente los faltantes por cero, porque *sin medición* no significa *cero vehículos*.

En las siguientes celdas se obtienen los **números y nombres reales del archivo**, que pueden variar según la estructura de sus hojas.

### 1.2 Localizar intervalos y columnas de conteos

Dado que las hojas de aforos pueden tener encabezados de varias filas, el siguiente código busca celdas que representen horas en la franja matutina y columnas numéricas con registros de volumen. El sistema muestra **candidatos preliminares**, no certifica por sí mismo qué giro representa cada columna.

Si la detección falla por un formato especial, se pueden ajustar los índices en la celda siguiente. La tabla que finalmente se analice **sale directamente del Excel**.

In [ ]:
# Funciones de exploración para hojas con encabezados no estándar
def leer_hora(valor):
    if pd.isna(valor):
        return None
    if isinstance(valor, (datetime, time, pd.Timestamp)):
        return f"{valor.hour:02d}:{valor.minute:02d}"
    if isinstance(valor, (int, float, np.number)) and not isinstance(valor, bool):
        n = float(valor)
        if 0 <= n < 1:
            minutos = round(n * 24 * 60) % (24 * 60)
            return f"{minutos//60:02d}:{minutos%60:02d}"
        if n.is_integer() and 600 <= n <= 1100:  # Horas en formato 700, 715...
            hh, mm = divmod(int(n), 100)
            if hh <= 23 and mm <= 59:
                return f"{hh:02d}:{mm:02d}"
        return None
    cadena = str(valor).strip()
    match = re.search(r"(?<!\d)(\d{1,2}):(\d{2})(?!\d)", cadena)
    if match:
        hh, mm = map(int, match.groups())
        return f"{hh:02d}:{mm:02d}" if hh < 24 and mm < 60 else None
    return None

def leer_volumen(valor):
    if pd.isna(valor) or isinstance(valor, bool):
        return np.nan
    if isinstance(valor, str):
        texto = valor.strip().replace(" ", "")
        if not texto or "%" in texto or texto in {"-", "--", "N/A", "NA"}:
            return np.nan
        texto = texto.replace(",", "")
        if not re.fullmatch(r"-?\d+(?:\.\d+)?", texto):
            return np.nan
        return float(texto)
    if isinstance(valor, (int, float, np.number)):
        return float(valor)
    return np.nan

def columna_excel(i):
    i += 1
    letras = ""
    while i:
        i, rem = divmod(i - 1, 26)
        letras = chr(65 + rem) + letras
    return letras

def nombre_de_columna(tabla, indice_columna, primera_fila):
    # Recupera encabezados inmediatos; no atribuye una dirección inexistente
    anteriores = tabla.iloc[max(0, primera_fila - 6):primera_fila, indice_columna]
    etiquetas = [str(x).strip().replace("\n", " ") for x in anteriores if pd.notna(x)]
    etiquetas = [x for x in etiquetas if x and len(x) <= 65]
    encabezado = " / ".join(dict.fromkeys(etiquetas[-2:])) if etiquetas else "sin encabezado identificable"
    return f"Col. {columna_excel(indice_columna)} — {encabezado}"

def detectar_tablas(hojas):
    opciones = []
    for nombre, tabla in hojas.items():
        for col_hora in range(min(tabla.shape[1], 55)):
            tiempos = tabla.iloc[:, col_hora].map(leer_hora)
            indices = [
                i for i, t in enumerate(tiempos)
                if t is not None and 6 <= int(t[:2]) <= 10
            ]
            if len(indices) < 4 or len(set(tiempos.iloc[indices])) < 4:
                continue
            fila_inicial = min(indices)
            columnas = []
            for j in range(tabla.shape[1]):
                if j == col_hora:
                    continue
                serie = pd.to_numeric(
                    tabla.iloc[indices, j].map(leer_volumen), errors="coerce"
                )
                validos = serie.dropna()
                if len(validos) < max(4, int(0.65 * len(indices))):
                    continue
                if validos.nunique() <= 1 or (validos < 0).any():
                    continue
                nombre_columna = nombre_de_columna(tabla, j, fila_inicial)
                if re.search(r"\b(total|suma|promedio|porcentaje|factor|fhm|phf)\b",
                             nombre_columna.lower()):
                    continue
                columnas.append((j, nombre_columna, float(validos.sum())))
            if len(columnas) >= 3:
                opciones.append({
                    "hoja": nombre,
                    "col_hora": col_hora,
                    "filas": indices,
                    "columnas": sorted(columnas, key=lambda elemento: elemento[2], reverse=True),
                    "puntaje": len(indices) * 4 + len(columnas)
                })
    return sorted(opciones, key=lambda opcion: opcion["puntaje"], reverse=True)

opciones = detectar_tablas(hojas_crudas)
print("Tablas candidatas encontradas:", len(opciones))
for k, opcion in enumerate(opciones[:5]):
    print(f"\nOpción {k}: hoja={opcion['hoja']!r}; tiempo=columna {columna_excel(opcion['col_hora'])}; "
          f"{len(opcion['filas'])} registros; {len(opcion['columnas'])} columnas numéricas candidatas")
    for j, etiqueta, suma in opcion["columnas"][:10]:
        print(f"  {columna_excel(j):>3}: {etiqueta[:75]} | suma preliminar={suma:.0f}")
if not opciones:
    print("\nNo fue posible reconocer automáticamente la tabla.")
    print("Revisa la vista inicial y configura los índices de hoja, hora y columnas en la siguiente celda.")

---

## Parte 2. Selección y visualización de movimientos vehiculares

### 2.1 Verificación de la selección

Se requiere **elegir tres movimientos distintos**. La detección automática propone las tres columnas candidatas con mayor volumen acumulado y muestra sus encabezados. Es necesario **comprobar en el croquis** si son tres movimientos independientes o, por el contrario, columnas de subtotales.

En caso de que las columnas seleccionadas no correspondan a tres movimientos, cambia \`COLUMNAS_MANUALES\` usando las letras que aparecen en la exploración. Por ejemplo: \`["D", "G", "K"]\`. También puedes escribir \`NOMBRES_MANUALES\` con los nombres exactos que logres identificar **sin inventar la correspondencia**.

Los valores de estas variables configuran la lectura; no cambian ni generan nuevos conteos.

In [ ]:
# CONFIGURACIÓN OPCIONAL: mantener None para usar la detección preliminar
OPCION_TABLA = 0               # Índice impreso en la celda anterior
HOJA_MANUAL = None             # Ejemplo: "Hoja1"
COLUMNA_HORA_MANUAL = None     # Letra de Excel, ejemplo "A"
COLUMNAS_MANUALES = None       # Tres letras, ejemplo ["D", "G", "K"]
NOMBRES_MANUALES = None        # Tres nombres comprobados en el croquis (opcional)

def indice_columna(letra):
    indice = 0
    for c in letra.strip().upper():
        if not ("A" <= c <= "Z"):
            raise ValueError(f"Letra de columna no válida: {letra}")
        indice = indice * 26 + (ord(c) - ord("A") + 1)
    return indice - 1

if opciones and HOJA_MANUAL is None:
    opcion = opciones[OPCION_TABLA]
    hoja_analisis = opcion["hoja"]
    columna_hora = opcion["col_hora"]
    filas = opcion["filas"]
    columnas_seleccionadas = [x[0] for x in opcion["columnas"][:3]]
else:
    if HOJA_MANUAL is None or COLUMNA_HORA_MANUAL is None:
        raise ValueError("Especifica HOJA_MANUAL y COLUMNA_HORA_MANUAL tras revisar los encabezados.")
    hoja_analisis = HOJA_MANUAL
    columna_hora = indice_columna(COLUMNA_HORA_MANUAL)
    tabla_ref = hojas_crudas[hoja_analisis]
    filas = [
        i for i, t in enumerate(tabla_ref.iloc[:, columna_hora].map(leer_hora))
        if t is not None and 6 <= int(t[:2]) <= 10
    ]
    columnas_seleccionadas = []

if COLUMNAS_MANUALES is not None:
    columnas_seleccionadas = [indice_columna(c) for c in COLUMNAS_MANUALES]

if len(columnas_seleccionadas) != 3 or len(set(columnas_seleccionadas)) != 3:
    raise ValueError("Selecciona exactamente TRES columnas distintas que representen movimientos.")
if len(filas) < 4:
    raise ValueError("No se localizaron suficientes intervalos matutinos; revisa la columna de hora.")

tabla_ref = hojas_crudas[hoja_analisis]
if NOMBRES_MANUALES is not None:
    if len(NOMBRES_MANUALES) != 3:
        raise ValueError("NOMBRES_MANUALES debe incluir exactamente tres descripciones.")
    nombres = list(NOMBRES_MANUALES)
else:
    nombres = [nombre_de_columna(tabla_ref, c, min(filas)) for c in columnas_seleccionadas]

print("Hoja analizada:", hoja_analisis)
print("Columna de hora:", columna_excel(columna_hora))
for k, (c, n) in enumerate(zip(columnas_seleccionadas, nombres), 1):
    print(f"Movimiento {k}: columna {columna_excel(c)} -> {n}")

if NOMBRES_MANUALES is None:
    print("\nATENCIÓN: verifica con el croquis los códigos/direcciones.")
    print("Si las columnas son subtotales o los nombres son ambiguos, ajusta COLUMNAS_MANUALES y NOMBRES_MANUALES.")

In [ ]:
# Convertir registros en una tabla analítica limpia: un renglón por intervalo
tiempos = [leer_hora(tabla_ref.iat[i, columna_hora]) for i in filas]
valores = {}
for k, col in enumerate(columnas_seleccionadas, 1):
    valores[f"Movimiento {k}"] = [leer_volumen(tabla_ref.iat[i, col]) for i in filas]

datos = pd.DataFrame({"Intervalo": tiempos, **valores})
datos["Orden"] = datos["Intervalo"].str[:2].astype(int) * 60 + datos["Intervalo"].str[3:5].astype(int)
datos = datos.sort_values("Orden").drop(columns="Orden").reset_index(drop=True)

movimientos = ["Movimiento 1", "Movimiento 2", "Movimiento 3"]
faltantes = datos[movimientos].isna().sum()
if datos[movimientos].notna().sum().min() < 4:
    raise ValueError("Al menos uno de los movimientos tiene menos de cuatro valores medidos.")

display(Markdown("### Tabla comparativa de tres movimientos"))
display(datos)
print("Registros:", len(datos))
print("Intervalos únicos:", datos["Intervalo"].nunique())
print("Valores faltantes por movimiento:")
display(faltantes.to_frame("Conteos faltantes"))

print("\nDescripción estadística por movimiento:")
display(datos[movimientos].describe().round(2))

# Los faltantes se dejan como NaN; no se sustituyen por cero
assert datos.shape[1] == 4

### 2.2 Correspondencia con el croquis

El croquis matutino muestra la rotonda y las trayectorias de sus accesos, entre ellos Av. Eugenio Garza Sada, Av. Fernando García Roel, Del Estado y el acceso de Tec Monterrey. También incluye conteos por trayectorias representadas con flechas.

La **relación exacta** entre los tres movimientos elegidos y las flechas del croquis depende de los nombres o códigos originales en \`Int. 01 mañana.xlsx\`. Las etiquetas que aparecen en las salidas anteriores proceden de los encabezados existentes. Si esos encabezados solo usan claves numéricas, se debe consultar su equivalencia antes de afirmar que una columna es un giro concreto.

En las visualizaciones se emplean las etiquetas conservadoras \`Movimiento 1\`, \`Movimiento 2\` y \`Movimiento 3\` para evitar atribuirles calles sin confirmación.

In [ ]:
# Gráfica 1: evolución por intervalo de los tres movimientos
fig, ax = plt.subplots(figsize=(11, 5))
for movimiento in movimientos:
    ax.plot(datos["Intervalo"], datos[movimiento],
            marker="o", linewidth=2, markersize=5, label=movimiento)
ax.set_title("Intersección 01 | Volumen por intervalo — tres movimientos (mañana)")
ax.set_xlabel("Inicio del intervalo de medición")
ax.set_ylabel("Vehículos por intervalo registrado")
ax.grid(alpha=0.25)
ax.legend(title="Serie del Excel")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 2: volúmenes totales observados en los intervalos del Excel
totales = datos[movimientos].sum(axis=0, skipna=True)
fig, ax = plt.subplots(figsize=(8.5, 4.5))
barras = ax.bar(movimientos, totales.values, color=["#2563eb", "#059669", "#d97706"])
for b, v in zip(barras, totales.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:,.0f}",
            ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, max(totales.values) * 1.17)
ax.set_ylabel("Vehículos registrados (suma de intervalos disponibles)")
ax.set_title("Comparación de los tres movimientos seleccionados")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Comparación de los movimientos: máximo por serie y máximo conjunto por intervalo
datos["Suma de los tres"] = datos[movimientos].sum(axis=1, min_count=1)
mov_mayor = totales.idxmax()
intervalo_max = datos.loc[datos["Suma de los tres"].idxmax(), "Intervalo"]
print("Mayor volumen acumulado:", mov_mayor, f"({totales[mov_mayor]:,.0f} vehículos)")
print("Intervalo con mayor suma entre las tres series:", intervalo_max)
print("NOTA: estos resultados corresponden SOLO a las tres columnas seleccionadas.")
print("No deben confundirse con la hora de máxima demanda de toda la intersección.")

### Interpretación 2. Comportamiento del tráfico

- **Movimiento con mayor volumen:** se identifica mediante la suma de sus registros en la tabla y la segunda gráfica. El resultado numérico aparece inmediatamente arriba.
- **Intervalo de mayor demanda:** se calcula como el intervalo con la mayor **suma de los tres movimientos seleccionados**, no como la hora de máxima demanda de toda la intersección. Es necesario diferenciar ambos conceptos.
- **¿Se comportan igual?** No debe suponerse que sí: la gráfica de líneas permite contrastar picos, disminuciones y variaciones entre series. Sus diferencias se cuantifican en la celda siguiente.
- **¿Por qué importan esas diferencias?** Si todos los movimientos recibieran tiempos de verde iguales, podrían producirse esperas excesivas en unos y tiempo desaprovechado en otros. Sin embargo, una mayor demanda por sí sola no determina un split: también importan la saturación, los conflictos, las colas y los peatones.

**Referencia adicional:** el croquis informa que la hora de máxima demanda matutina del conjunto de la intersección fue 08:00–09:00, con 1,771 vehículos; este valor no necesariamente coincide con el máximo puntual de nuestras tres series.

In [ ]:
# Interpretación numérica directamente vinculada con los datos procesados
resumen = pd.DataFrame({
    "Volumen total": totales,
    "Máximo por intervalo": datos[movimientos].max(),
    "Intervalo del máximo": [
        datos.loc[datos[m].idxmax(), "Intervalo"] for m in movimientos
    ],
    "Desviación estándar": datos[movimientos].std()
})
display(resumen.round(2))
orden = totales.sort_values(ascending=False)
print(f"El movimiento de mayor volumen es {orden.index[0]} ({orden.iloc[0]:,.0f} vehículos).")
print(f"El de menor volumen es {orden.index[-1]} ({orden.iloc[-1]:,.0f} vehículos).")
print(f"La mayor suma de las tres series se produce en el intervalo {intervalo_max}.")
print("La desviación estándar resume la variabilidad observada de cada movimiento.")

---

## Parte 3. Normalización de los datos

### Transformación Min-Max

Se usa la función, calculada **por separado para cada movimiento**, sobre los intervalos efectivamente disponibles:

\[
x_{\text{normalizado}}=\frac{x-x_{\min}}{x_{\max}-x_{\min}}.
\]

Los valores originales se conservan para mostrar las unidades de tráfico. La escala transformada sirve únicamente para **comparar formas y cambios relativos** en una misma gráfica.

**Volumen constante:** si \(x_{\max}=x_{\min}\), la expresión tiene denominador cero. Para fines de esta actividad, todos los registros válidos de esa columna se asignan a 0 y se deja constancia de que la serie no presenta variación. No se convierten en cero los registros faltantes.

In [ ]:
# Normalización Min-Max por serie, con manejo explícito de constantes y NaN
normalizados = pd.DataFrame(index=datos.index)
constantes = []
for movimiento in movimientos:
    serie = datos[movimiento].astype(float)
    minimo, maximo = serie.min(), serie.max()
    if pd.isna(minimo):
        raise ValueError(f"El {movimiento} no tiene valores válidos.")
    if np.isclose(maximo, minimo):
        normalizados[movimiento] = serie.apply(lambda x: np.nan if pd.isna(x) else 0.0)
        constantes.append(movimiento)
    else:
        normalizados[movimiento] = (serie - minimo) / (maximo - minimo)

tabla_normalizacion = pd.DataFrame({"Intervalo": datos["Intervalo"]})
for movimiento in movimientos:
    tabla_normalizacion[f"{movimiento} original"] = datos[movimiento]
    tabla_normalizacion[f"{movimiento} Min-Max"] = normalizados[movimiento]

display(Markdown("### Tabla de valores originales y normalizados"))
display(tabla_normalizacion.round(4))
print("Movimientos constantes:", constantes if constantes else "ninguno")

valores_normalizados = normalizados.to_numpy(dtype=float)
assert np.all((valores_normalizados[np.isfinite(valores_normalizados)] >= -1e-10))
assert np.all((valores_normalizados[np.isfinite(valores_normalizados)] <= 1 + 1e-10))

In [ ]:
# Gráfica 3: comparar las tres series normalizadas
fig, ax = plt.subplots(figsize=(11, 5))
for movimiento in movimientos:
    ax.plot(datos["Intervalo"], normalizados[movimiento],
            marker="o", linewidth=2, markersize=5, label=movimiento)
ax.set_title("Intersección 01 | Volúmenes normalizados con Min-Max")
ax.set_xlabel("Inicio del intervalo de medición")
ax.set_ylabel("Volumen normalizado (0 a 1)")
ax.set_ylim(-0.06, 1.07)
ax.grid(alpha=0.25)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("La primera gráfica muestra vehículos por intervalo; esta tercera gráfica muestra escala relativa.")

### Interpretación 3. Normalización

- **¿Qué cambió?** El volumen dejó de expresarse en vehículos y se transformó a una escala numérica de 0 a 1 **para cada movimiento por separado**. El mínimo observado se representa con 0 y el máximo con 1, salvo en series constantes.
- **¿Se conservan los patrones?** Sí: cuando el rango no es cero, Min-Max es una transformación lineal creciente, así que conserva el orden temporal, la ubicación de máximos/mínimos y la forma de crecimiento o disminución, aunque cambia la escala vertical.
- **¿Por qué sirve antes de una MLP?** Permite que magnitudes distintas tengan rangos comparables. Puede mejorar la estabilidad y eficiencia del ajuste de los pesos y evitar que una entrada numéricamente grande domine solo por su escala.
- **¿Todos tienen el mismo volumen?** No. Un 1 en dos series diferentes puede representar volúmenes absolutos distintos. Es necesario consultar los datos originales para comparar cantidades de vehículos.
- **¿Qué pasa si hay un movimiento constante?** Su valor normalizado se fija en 0 y se documenta. Una serie constante no aporta variación en estos registros.

**Prevención de fuga de información:** esta normalización usa los registros para una exploración didáctica. En una MLP real, mínimos y máximos se calculan únicamente con el conjunto de entrenamiento, y después se aplican sin reajuste a validación, prueba y operación.

---

## Parte 4. Identificación de patrones para una futura MLP

### Propuesta de variables de entrada

| Variable candidata | Información que aportaría |
|---|---|
| Volumen de movimiento 1, vehículos por intervalo | Cuánta demanda llega a esa trayectoria durante un periodo definido. |
| Volumen de movimiento 2, vehículos por intervalo | Demanda concurrente que puede competir por tiempo de servicio. |
| Volumen de movimiento 3, vehículos por intervalo | Diferencias de carga entre movimientos y posibles cambios de prioridad. |
| Hora del día codificada numéricamente | Patrones horarios de llegada (preferiblemente codificación cíclica para todo el día). |
| Ocupación de carril y longitud de cola (datos adicionales) | Estado de congestión que los conteos de paso, por sí solos, no muestran. |

### Relaciones que podría aprender

La red podría identificar franjas con más circulación, variaciones de demanda entre trayectorias y relaciones entre volumen, ocupación y acumulación de colas. Para que prediga **tiempos de verde NS/EO o splits por fase**, harían falta pares consistentes de **entradas** (estado del tráfico) y **objetivos** (tiempos de verde de referencia válidos o determinados mediante una política optimizada).

No sería metodológicamente correcto fabricar etiquetas de tiempos de verde a partir de estos conteos sin un criterio de control y sin comprobar la seguridad y las restricciones de la intersección.

### Interpretación 4. Conexión con las redes neuronales

- **¿Qué datos del estudio usaría como entradas?** Los tres volúmenes por intervalo seleccionados, sus marcas de tiempo y, si existen en otras columnas debidamente documentadas, más movimientos del mismo instante. También buscaría incorporar ocupación, longitud de cola y flujos peatonales.
- **¿Qué me gustaría predecir?** Duraciones de verde por fase o dirección que sean seguras, operativamente factibles y útiles para disminuir las demoras.
- **¿Contamos con las etiquetas objetivo?** No se han identificado en estos datos de conteo etiquetas verificadas de tiempos de verde recomendados. Para entrenar un modelo supervisado de regresión se necesitarían esas etiquetas y el método usado para obtenerlas.
- **¿Observar un patrón equivale a que la red lo aprendió?** No. Una gráfica permite describir datos ya observados. Para afirmar que un modelo aprendió una relación, debería entrenarse con ejemplos adecuados y mantener capacidad predictiva en otros intervalos o días no utilizados en el ajuste.
- **¿Cómo comprobaría que generaliza?** Separaría los datos por tiempo o por día, reservaría un conjunto de prueba no visto, ajustaría el escalador solo con entrenamiento y reportaría MAE/RMSE de los splits. Después compararía las estrategias de control en una simulación calibrada y analizaría demora, colas, detenciones y seguridad.

La MLP no se implementa ni entrena en esta actividad, conforme a las instrucciones.

### Verificación final de la exploración
Las comprobaciones permiten confirmar que el análisis se realizó sobre tres columnas de datos cargados del Excel. Un resultado correcto **no garantiza por sí solo** que la correspondencia espacial con el croquis haya sido validada: esa revisión depende del significado de los encabezados originales.

In [ ]:
# Resumen técnico reproducible del análisis
assert len(movimientos) == 3
assert len(columnas_seleccionadas) == 3
assert datos["Intervalo"].notna().all()
assert set(movimientos).issubset(datos.columns)
assert set(movimientos).issubset(normalizados.columns)

print("Archivo fuente:", ARCHIVO)
print("Hoja de procedencia:", hoja_analisis)
print("Registros analizados:", len(datos))
print("Columnas del Excel:", [columna_excel(c) for c in columnas_seleccionadas])
print("Valores faltantes:", faltantes.to_dict())
print("Mayor suma de los tres movimientos por intervalo:", intervalo_max)
print("Mayor total de un movimiento:", mov_mayor)
print("Estado: análisis descriptivo completado; ninguna MLP ha sido entrenada.")

---

## Conclusión: ¿qué aprendí sobre los datos de tráfico?

Esta actividad me permitió comprender que, antes de diseñar una red neuronal para el control semafórico, es necesario examinar cómo están organizados los datos y qué representa cada medición. El croquis de la intersección 01 muestra que la demanda no permanece constante a lo largo del día: durante las horas de máxima demanda se registran 1,771 vehículos por la mañana, 2,287 por la tarde y 2,372 por la noche. Sin embargo, para estudiar movimientos individuales es indispensable revisar los intervalos del archivo Excel y no utilizar esos totales generales como si fueran conteos de cada giro.

Al comparar tres movimientos, las tablas y gráficas permiten identificar diferencias de volumen y momentos de mayor actividad. La normalización Min-Max facilita comparar sus variaciones relativas en una escala común, pero no significa que transporten la misma cantidad de vehículos.

También comprendí que encontrar patrones en los datos no equivale a construir un modelo confiable. Para el reto de SEMEX será necesario disponer de más observaciones, conocer las restricciones del semáforo y obtener tiempos de verde objetivo debidamente justificados. Solo después tendría sentido entrenar una MLP y evaluar si sus recomendaciones realmente reducen las demoras.